# Lesson 12 · A matrix isn't a table of numbers, it's a transformation machine — matrix multiplication and linear transformations

**Goal**: matrix times vector = rotating/stretching the vector; matrix times matrix = chaining several transformations together.

**Why it matters for Aurora**: the mel filter matrix `(M, N)` compresses an FFT spectrum `(N,)` into a mel spectrum `(M,)` — that's a single `matvec`; the naive DFT implementation `dft()` is likewise an N×N complex matrix multiplying a signal vector.

← **Previous lesson** · [L11 · Vector norms](L11_norms.ipynb)

> Last lesson covered **vector norms**: three rulers (L1 / L2 / ∞) for measuring a vector, and the geometry of normalization.
> This lesson explores **matrix multiplication**.

## This lesson's story: a matrix is a coordinate-transformation machine

Ever used the zoom on a camera? Press it and the picture gets bigger — every pixel is multiplied by the same factor; the direction doesn't change, things just stretch. That's what a diagonal matrix does.

Now imagine rotating the camera: the picture turns — every point rotates about the origin by the same angle. That's a rotation matrix.

A matrix isn't a "table of numbers"; it's a **coordinate-transformation machine**:
```
input vector x (shape n) → matrix A (shape m×n) → output vector y (shape m)
```

This machine has a few properties that belong only to linear transformations:
- **Superposition**: `A(u + v) = Au + Av` (add then transform = transform then add)
- **Scaling**: `A(cu) = c·Au` (scale then transform = transform then scale)
- **Composition**: `A @ (B @ x) = (A @ B) @ x` (B first, then A = just use AB)

In this lesson you'll write `matvec` (matrix × vector) and `matmul` (matrix × matrix) yourself with loops, and understand the "row-by-row dot product" behind multiplication.

> **Main path**: matvec → the "linear combination of columns" picture → matmul → composition.
> The long asides in between (why linearity comes built in, where the dot-product rule comes from…) are for going deeper — **implement first, then dig in**.
> Writing the double loop by hand is for **understanding**; in production just use `A @ x` / `A @ B`.

### Aside: why do "superposition" and "scaling" come built into matrices?

Picture the "recipe card" of a vending machine: how many of each kind of coin you put in (the input) gets converted, at fixed ratios, into how many of each item comes out (the output). This machine has a very honest "personality" — it uses the same fixed weights for every coin, and doesn't quietly change the recipe just because you put in a few extra coins.

That honesty, written as maths, says: if you convert two batches of coins separately and add the results, you get exactly the same as merging the two batches first and then converting — because all the recipe card ever does is "number of each kind of coin × fixed weight, then add up," and multiplication distributes over addition (`a×(x+y) = a×x + a×y`, the distributive law you learned in elementary school). The "row-by-row dot product" of matrix multiplication is essentially lots of these "recipe-card conversions" stacked together, so:

- **Superposition**: `A(u+v) = Au+Av` — not some special power of matrices; it's just the distributive law holding once in each row, so it still holds when you collect them all.
- **Scaling**: `A(cu) = c·Au` — same idea: multiply every input by the same c, every term in the dot product gets multiplied by c too, and you can pull c outside.

Let's look at a concrete example instead of abstract symbols: let A=[[1,2],[3,4]], u=[1,0], v=[0,1].

```
A@u = [1,3]        (picks out A's first column)
A@v = [2,4]        (picks out A's second column)
A@u + A@v = [3,7]

u+v = [1,1]
A@(u+v) = [1×1+2×1, 3×1+4×1] = [3,7]   ← same as above!
```

Together these two properties have a technical name, "linearity" — but now you know they aren't extra magic bestowed on matrices; they fall out automatically from the definition "row-by-row dot product" plus the ordinary distributive law. The code below checks this concrete example.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np

A = np.array([[1.0, 2.0], [3.0, 4.0]])
u = np.array([1.0, 0.0])
v = np.array([0.0, 1.0])

lhs = A @ (u + v)          # add first, then transform
rhs = A @ u + A @ v        # transform first, then add
print('A @ (u+v)   =', lhs)
print('A@u + A@v   =', rhs)
assert np.allclose(lhs, rhs)

c = 3.0
lhs2 = A @ (c * u)
rhs2 = c * (A @ u)
print('A @ (3u)    =', lhs2)
print('3 * (A@u)   =', rhs2)
assert np.allclose(lhs2, rhs2)
print('\n✅ Superposition and scaling both check out — a natural consequence of the distributive law, not extra matrix magic.')


## 1. Matrix × vector: a row-by-row dot product

The shape `(m, n)` of a matrix A tells you two things:
- Input: a vector of length n (n components go in)
- Output: a vector of length m (m components come out)

How to compute it: **row i of the output = the dot product of row i of A with x**.

A worked example by hand: A = [[1, 2], [3, 4]], x = [5, 6]

```
y[0] = 1×5 + 2×6 = 5 + 12 = 17
y[1] = 3×5 + 4×6 = 15 + 24 = 39
```

So `A @ [5, 6] = [17, 39]`.

NumPy check: `np.array([[1,2],[3,4]]) @ np.array([5,6])` = `array([17, 39])`.

This "row-by-row dot product" is the internal logic of the `matvec` function — in this lesson you'll write it yourself with a loop.

### Aside: where does this "row-by-row dot product" rule actually come from?

You might ask: the determinant you may have met at school also involves a matrix — is that how multiplication worked there? — In fact the determinant is something completely different (it computes a single number from a matrix; it isn't a matrix times a vector). "Matrix × vector" here is a separate set of rules, designed specifically so that a matrix can represent a "coordinate-transformation machine."

An analogy for the motivation behind this design: think of the matrix as a "recipe book" with m pages (rows). Each page is an independent recipe telling you "look at the n input numbers, multiply each by some weight, and add them up" to produce one number of the output. Page i only computes output number i, and doesn't interfere with the other pages.

This design has one big advantage (proved in section 5): if matrix B stands for "do one transformation first" and matrix A for "then do another," the whole process "B first, then A" can be packed into *another* matrix, `A@B`, that does it in one go. Only the "row-by-row dot product" definition makes this "packing" possible — and that's the fundamental reason mathematicians chose to define matrix multiplication this way: not an arbitrary convention, but so that matrix multiplication equals "function composition" (proved concretely in section 5 below).

So "why is the output m-dimensional rather than n-dimensional?" — the answer is hidden in the phrase "the recipe book has m pages":
- The matrix has m rows ⟹ there are m recipes ⟹ they produce m output numbers. The number of rows directly decides the output dimension, because each row produces one output component.
- Every recipe page has to read all n input numbers to get its result (a dot product needs both sides to be the same length) ⟹ the input must be n-dimensional, matching the number of columns of the matrix.

So "rows = output dimension, columns = input dimension" isn't a memory rule that happens to work; it's dictated by the structure "each row is an independent recipe, and each recipe consumes the whole input." The reverse, "columns = output dimension," makes no sense, because what decides "how many outputs there are" is "how many recipes there are" (rows), not "how many numbers each recipe uses" (columns).

## Notation first: look at the shape, then the operation

Every object in linear algebra has a definite shape: a vector is `(n,)`, a matrix is `(m, n)`, and matrix-times-vector turns `(n,)` into `(m,)`. For every example, first mark the input and output dimensions, then look at the actual numbers.

In [ ]:
import numpy as np
W = np.array([[2.0, 0.0],
              [0.0, 3.0]])   # stretches x by 2 and y by 3
x = np.array([1.0, 1.0])
print('W @ x =', W @ x)     # [2. 3.]

## Hands-on: for linear-algebra objects, look at the shape first, then the meaning

Run the code below and notice the shape before and after `A @ v`: the input `v` is `(2,)` and the output is still `(2,)`, but the diagonal entries of the matrix `A` decide how much each component is stretched.

In [ ]:
import numpy as np

v = np.array([3.0, 4.0])
A = np.array([[2.0, 0.0],
              [0.0, 0.5]])

print('v =', v, 'shape =', v.shape)
print('A =')
print(A)
print('A shape =', A.shape)
print('A @ v =', A @ v)
print('vector length ||v|| =', np.linalg.norm(v))


## Code experiment: loop over a few vectors and watch how the matrix changes them

Feed several vectors pointing in different directions into the same matrix, and watch how the direction and length of the output change with the input direction.

In [ ]:
import numpy as np

A = np.array([[2.0, 1.0],
              [0.0, 1.0]])
vectors = [np.array([1.0, 0.0]), np.array([0.0, 1.0]), np.array([1.0, 1.0]), np.array([2.0, -1.0])]

print('A =')
print(A)
for v in vectors:
    out = A @ v
    print(f'v={v} -> A@v={out}')


## 2. Rotation matrices: turning a vector through an angle

### Aside: how did cos and sin get into the rotation matrix?

First recall something from school: on a circle of radius 1 (the unit circle), draw an arrow from the center pointing at angle φ; the tip of this arrow has coordinates `(cos φ, sin φ)` — this is really the original definition of cos and sin ("adjacent/hypotenuse" and "opposite/hypotenuse" in a right triangle; when the hypotenuse has length 1, they're simply the coordinates).

Now imagine turning this arrow — not turning the coordinate system, but rotating the arrow itself about the center by an angle θ. The arrow pointed at angle φ before; afterwards it points at angle `φ+θ`. By the same definition, the new tip has coordinates `(cos(φ+θ), sin(φ+θ))`.

The angle-addition formulas from school (if you've forgotten them, just take the result on trust for now — it won't get in the way):

```
cos(φ+θ) = cosφ·cosθ − sinφ·sinθ
sin(φ+θ) = sinφ·cosθ + cosφ·sinθ
```

Now relax "radius 1" to "radius r," with the original point at `(x, y) = (r·cosφ, r·sinφ)`. Substitute into the two formulas above:

```
x' = r·cos(φ+θ) = r·cosφ·cosθ − r·sinφ·sinθ = x·cosθ − y·sinθ
y' = r·sin(φ+θ) = r·sinφ·cosθ + r·cosφ·sinθ = x·sinθ + y·cosθ
```

In other words, after any point `(x, y)` is rotated about the origin by an angle θ, its new coordinates are `(x·cosθ − y·sinθ,  x·sinθ + y·cosθ)`. Lay these two equations out as a matrix times a vector:

```
[x']   [cosθ  −sinθ] [x]
[y'] = [sinθ   cosθ] [y]
```

That's exactly the rotation matrix `R = [[cosθ, −sinθ], [sinθ, cosθ]]` — cos on the diagonal and sin with a minus sign in the top right aren't arranged at random; they're the positions that fall out naturally when you expand the angle-addition formulas.

(**Optional, if you know complex numbers**: this is really the matrix version of complex multiplication by `e^{iθ} = cosθ + i·sinθ` — represent "multiply by e^{iθ}" as a matrix and you get exactly this rotation matrix. If you don't know complex numbers, it won't stop you carrying on.)

**Let's check this agrees with the "linear combination of columns" view**: the first column of the matrix should equal `R @ [1,0]` — plugging x=1, y=0 into the formula gives `(cosθ, sinθ)`, which is exactly R's first column; the second column should equal `R @ [0,1]` — plugging in x=0, y=1 gives `(−sinθ, cosθ)`, exactly R's second column. The two views agree.

**Quick reference for special angles** (you'll need it for the whiteboard challenge — worth memorizing or bookmarking):

| θ | 0° | 30° | 45° | 60° | 90° | 180° | 270° |
|---|---|---|---|---|---|---|---|
| cos θ | 1 | √3/2 ≈ 0.866 | √2/2 ≈ 0.707 | 1/2 | 0 | −1 | 0 |
| sin θ | 0 | 1/2 | √2/2 ≈ 0.707 | √3/2 ≈ 0.866 | 1 | 0 | −1 |

**Worked demo by hand (we go through all the steps once with 45° to get you used to the routine; for the 90° in the whiteboard challenge, repeat the process yourself)**:

1. Look up the table: `cos45° = sin45° ≈ 0.707`
2. Plug into the matrix: `R(45°) = [[0.707, −0.707], [0.707, 0.707]]`
3. The vector to rotate is `x = [3, 0]` (the code below draws it)
4. Row-by-row dot products: `y[0] = 0.707×3 + (−0.707)×0 = 2.121`; `y[1] = 0.707×3 + 0.707×0 = 2.121`
5. So `R(45°) @ [3,0] ≈ [2.121, 2.121]` — the code below draws this arrow and confirms it really has turned to the 45° direction.

In [ ]:
import matplotlib.pyplot as plt
theta = np.pi/4  # 45°
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
x = np.array([3.0, 0.0]); xr = R @ x
plt.figure(figsize=(4,4))
plt.quiver(0,0,x[0],x[1], angles='xy',scale_units='xy',scale=1,color='gray')
plt.quiver(0,0,xr[0],xr[1],angles='xy',scale_units='xy',scale=1,color='red')
plt.xlim(-1,4);plt.ylim(-1,4);plt.grid(True,alpha=.3)
plt.gca().set_aspect('equal');plt.title('rotation by 45°');plt.show()

## 3. ✏️ Your task: write matrix × vector `matvec` yourself

No `@` allowed — spell out the computation with a loop.

**Line of reasoning**:
1. Output length = number of rows `m`; start with an array of zeros: `out = np.zeros(A.shape[0])`
2. For each row index `i`, compute the dot product of row `i` with `x` and store it in `out[i]`
3. Use `np.dot(A[i], x)` for the single-row dot product — it's equivalent to Σⱼ A[i,j]·x[j], so no inner loop is needed

> 🔍 **Notation tip**: don't panic when you see a summation like `Σⱼ A[i,j]·x[j]` — it's just shorthand for "multiply each element of row i by the element in the matching position of x, then add all the products up," a symbol mathematicians invented because they were tired of writing it out. It's exactly the same thing as the "dot product" you've already computed many times, not a new concept: `np.dot(A[i], x)` does precisely this in one line.

**Reference input and output**: `A=[[1,0],[0,2]], x=[3,4]` → `[1×3+0×4, 0×3+2×4] = [3, 8]`

> 💡 Need a hint? Once you've finished the exercise, you can compare with the reference implementation in the `solutions/` folder.


### Before you write code, fill in the variable table

Three things to pin down before you write `matvec`:
- Input: a matrix `A` (shape `(m, n)`) and a vector `x` (shape `(n,)`)
- Key step: for each row index `i`, compute `np.dot(A[i], x)` to get element `i` of the output
- Return: a vector of shape `(m,)`, numerically the same as `A @ x`

In [ ]:
def matvec(A, x):
    # ✏️ TODO: return A @ x using a loop (each element is the dot product of one row with x)
    out = np.zeros(A.shape[0])
    # for i in range(A.shape[0]):
    #     out[i] = ...
    raise NotImplementedError("TODO")


In [ ]:
A = np.array([[1.0,2.0],[3.0,4.0]])
x = np.array([5.0,6.0])
try:
    result = matvec(A, x)
    print('my matvec :', result)
    print('numpy  A@x :', A @ x)
    assert np.allclose(result, A @ x), 'should match A@x'
    print('\n✅ Passed: matrix times vector = a set of dot products.')
except (NotImplementedError, TypeError):
    print('⬜ matvec not implemented yet — please complete the TODO')


**🔗 Aurora connection**: `mel_filterbank()` in `src/aurora/audio/mel.py` returns a filterbank matrix of shape `(M, N)`, and the actual code of `mel_spectrogram()` is `return power @ fb.T`, where `power` has shape `(n_frames, N)` and `fb.T` is `(N, M)`, giving `(n_frames, M)`. This is equivalent to running `matvec(fb, frame)` on each frame separately, but with the frame axis placed along the rows and processed as one batch — which is why `fb` has to be transposed. The core of `dft()` in `src/aurora/audio/transforms.py` is `twiddle @ x`, an N×N complex matrix times the signal vector — both are direct instances of `matvec`.

> 🤔 **Why not write a for loop and call `matvec` once per frame?** — Actually that would give exactly the same numbers; it would just be slow. Picture a factory line: moving one box at a time (calling matvec in a loop) certainly gets the job done, but if you put many boxes on the conveyor belt at once and move them together (one matrix multiplication), the machinery makes fewer movements overall and it's faster. `power @ fb.T` treats "many frames" as one whole batch on the conveyor belt and computes it all in one go, rather than hauling one frame at a time with a slow Python for loop. This doesn't need any knowledge beyond this lesson — it's exactly the `matmul` you'll implement in section 4: each row of `power` (n_frames rows, N columns) is a frame, each column of `fb.T` (N rows, M columns) is a mel filter, and element (frame, m) of `power @ fb.T` = the dot product of row `frame` of `power` (one frame) with column m of `fb.T` (which is row m of `fb`) = component m of `matvec(fb, that frame)`. In other words, each row of the result matrix is exactly what you get by calling `matvec(fb, frame)` on that frame — `matmul` has already done "call matvec on lots of vectors separately, then lay the results out row by row" in one shot. A small example below confirms this.

In [ ]:
import numpy as np

# Mock up a small "fb" filter matrix (M=2 filters, N=3 frequency points)
fb_demo = np.array([[1.0, 0.0, 1.0],
                     [0.0, 1.0, 1.0]])

# Mock up a "power" matrix of 4 frames, 3 frequency points each
power_demo = np.array([[1.0, 2.0, 3.0],
                       [4.0, 5.0, 6.0],
                       [0.0, 1.0, 0.0],
                       [2.0, 2.0, 2.0]])

# Method 1: loop over the frames with a matvec-style computation (np.dot = row-by-row dot product)
loop_result = np.zeros((power_demo.shape[0], fb_demo.shape[0]))
for i, frame in enumerate(power_demo):
    loop_result[i] = np.array([np.dot(fb_demo[m], frame) for m in range(fb_demo.shape[0])])

# Method 2: one batched matrix multiplication (how mel_spectrogram is written in Aurora)
batch_result = power_demo @ fb_demo.T

print('frame-by-frame loop:\n', loop_result)
print('batched matrix multiplication:\n', batch_result)
assert np.allclose(loop_result, batch_result)
print('\n✅ The two methods give exactly the same numbers — the batched multiplication just "packs up" the loop and hands it to numpy, which is faster.')


### Aside: row-by-row dot products and a linear combination of columns — why are they the same thing?

By now you've implemented `matvec` with "row-by-row dot products." Next you'll meet another way of looking at matrix-times-vector: "treat the columns of the matrix as a set of building blocks, use the components of the input vector as weights, and assemble the weighted blocks." At first sight these two descriptions look like different things, but they're the same computation described from a different angle.

Work through the algebra once and it becomes clear. By definition, component i of the output is:

```
y[i] = Σⱼ A[i,j] · x[j]   (row-by-row dot-product view: row i · x)
```

Now look at this sum in a different order: instead of grouping by "row," group by "column j" — fix j and look at x[j]'s contribution to *all* output components. x[j]'s contribution to output i is `A[i,j] · x[j]`; taken over all i together, that's column j of A multiplied as a whole by the scalar x[j], giving the vector `x[j] · A[:,j]`. Add up these contribution vectors for j=0,1,...,n-1:

```
Σⱼ x[j] · A[:,j]   (linear-combination-of-columns view)
```

Expand this "adding up" for each component i, and you get exactly `Σⱼ x[j]·A[i,j]` — the same sum as the `y[i]` given by "row-by-row dot products," just written in a different order (group by j first, then expand by i, versus group by i first, then sum over j). So:

```
A @ x  =  Σⱼ x[j] · A[:,j]
```

Check it with a concrete example: A=[[1,2],[3,4]], x=[5,6].

- Row-by-row dot products: `y[0] = 1×5+2×6 = 17`, `y[1] = 3×5+4×6 = 39`
- Linear combination of columns: `x[0]·A[:,0] + x[1]·A[:,1] = 5·[1,3] + 6·[2,4] = [5,15] + [12,24] = [17,39]`

The two answers are identical. The code below runs both algorithms to confirm they agree for any input.

In [ ]:
import numpy as np

A = np.array([[1.0, 2.0], [3.0, 4.0]])
x = np.array([5.0, 6.0])

# View 1: row-by-row dot products
row_view = np.array([np.dot(A[i], x) for i in range(A.shape[0])])

# View 2: linear combination of columns — use each component of x as a weight and sum the weighted columns
col_view = np.zeros(A.shape[0])
for j in range(A.shape[1]):
    col_view += x[j] * A[:, j]

print('row-by-row dot-product view:', row_view)
print('linear-combination-of-columns view:', col_view)
assert np.allclose(row_view, col_view)
print('\n✅ Both views give exactly the same result — they are the same sum, grouped in a different order.')


## 🎨 Picture: matrix × vector = a linear combination of columns

The previous cell proved with algebra that "row-by-row dot products" and "a linear combination of columns" are the same thing; here we draw it. The figure shows `A @ x` both ways: on top, each row of A dotted with x gives one entry of the output; below, each column of the matrix is scaled by the matching component of the input vector x ("how many times longer or shorter this column gets"), and the scaled columns are added up — the sum is `A@x`.

> ⚠️ **If the line `from aurora.laviz import ...` raises an error in your environment** (usually a package that isn't installed properly, or an import-path problem): it doesn't affect your understanding of this lesson — you can skip this cell and carry on. The text version is the paragraph above — the figure shows the two column vectors `[1,3,5]` and `[2,4,6]` of the matrix `[[1,2],[3,4],[5,6]]`, stretched by the two components 2 and 1 of the input vector `[2,1]` respectively, then added up; the sum is the result of `matvec([[1,2],[3,4],[5,6]], [2,1])`, which is what `mat_times_vec([[1,2],[3,4],[5,6]], [2,1])` visualizes.

In [ ]:
from laviz import style, mat_times_vec
style()
mat_times_vec([[1,2],[3,4],[5,6]],[2,1]);

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('How will matrix A move these vectors?')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | length {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## Parameter experiment: a diagonal matrix and an all-ones matrix

Build the diagonal matrix `np.diag([2,3])`, feed in `[1,1]`, and confirm the output is `[2,3]` — row 0's weights `[2,0]` only pick up x[0], row 1's weights `[0,3]` only pick up x[1], so each dimension is scaled independently. Then swap in the all-ones square matrix `np.ones((2,2))`, feed in the same `[1,1]`, and watch the output become `[2,2]` — each row `[1,1]` dotted with `[1,1]` = 1+1 = 2, so each output entry is the sum of all the input elements.

> 🔍 **Why does a diagonal matrix have this "each minds its own business" property?** It's no coincidence, and no extra geometric magic either — it follows straight from the definition "row-by-row dot product." Recall: dotting row i with x means multiplying each element of row i by the element in the matching position of x and adding up. Row i of a diagonal matrix looks like this: everything is 0 except position i, which holds the diagonal entry. Whenever one factor of a product is 0, that term contributes 0 — so every term in the dot product except term i vanishes automatically, leaving only "the diagonal entry × x[i]." That's why a diagonal matrix behaves as "independent per-component scaling": not because it's especially clever, but because of the plainest multiplication rule, "the other entries are 0 ⟹ those terms in the dot product become zero."

## 4. ✏️ Going further: write matrix × matrix `matmul` yourself

You've just implemented matrix × vector; matrix × matrix is just one `matvec` for each column of B.

**Line of reasoning**: `C = A @ B`, `C[:, j] = matvec(A, B[:, j])` — call `matvec` once for each column of B, and put the results together to get C.

**Hint**: call `matvec(A, col)` on each column of B, then assemble all the result columns into a matrix (you choose how to stack them).

### Aside: why can matrix × matrix be "split into one matvec per column"?

First let's see why this is true mathematically, rather than "the course says so, just do it." The definition of matrix-times-matrix `C = A@B` (a natural extension of the "row-by-row dot product" idea): the number in row i, column j of `C` is the dot product of row i of A with column j of B:

```
C[i, j] = Σₖ A[i, k] · B[k, j]
```

Now focus on "column j" — fix j and treat `C[:, j]` (the whole of column j of C) as a vector. Its component i is `Σₖ A[i, k]·B[k, j]` — and that's exactly "row i of A dotted with the vector `B[:, j]`," which is precisely the definition of component i of `matvec(A, B[:, j])`!

So: `C[:, j] = matvec(A, B[:, j])` — for every fixed column index j. It's not a coincidence; it's what you get directly by looking at the definition of `C=A@B` column by column. That's why "matrix times matrix = one matrix-times-vector for each column of B, with the results put back together column by column."

**When you implement it, how do you put the computed columns back into a matrix?** NumPy has several "stacking" functions that are easy to mix up; here's how they differ:

- `np.column_stack([c1, c2, ...])`: stacks a list of 1D vectors as **columns** into a 2D matrix — exactly what we want.
- `np.stack([c1, c2, ...], axis=1)`: the more general version; `axis=1` means "stack along a new second dimension," with the same effect as `column_stack`.
- `np.hstack([c1, c2, ...])`: for 1D arrays this is "horizontal concatenation," and the result is still **one longer 1D vector**, not a matrix! This is the easiest trap to fall into — if you accidentally use it as a column-stacking function, `matmul` will output the wrong shape.
- `np.concatenate`: the lowest-level general function; you'd have to reshape each column to `(m, 1)` by hand first before concatenating, to get a matrix of the right shape.

**Self-check trick**: after stacking, check with `.shape` — `C` should be `(A.shape[0], B.shape[1])`. If the shape is wrong (it came out one-dimensional, say), you've most likely picked the wrong stacking function. The code below compares how these stacking options behave.

In [ ]:
import numpy as np

c1 = np.array([1.0, 2.0])
c2 = np.array([3.0, 4.0])

print('column_stack:\n', np.column_stack([c1, c2]), '  shape =', np.column_stack([c1, c2]).shape)
print('stack(axis=1):\n', np.stack([c1, c2], axis=1), '  shape =', np.stack([c1, c2], axis=1).shape)
print('hstack (the trap — not a matrix!):', np.hstack([c1, c2]), '  shape =', np.hstack([c1, c2]).shape)
print('concatenate + reshape:\n',
      np.concatenate([c1.reshape(-1, 1), c2.reshape(-1, 1)], axis=1),
      '  shape =', np.concatenate([c1.reshape(-1, 1), c2.reshape(-1, 1)], axis=1).shape)

# Self-check: a correctly stacked matrix should have shape (rows, columns)
correct = np.column_stack([c1, c2])
assert correct.shape == (2, 2), 'column_stack should give a (2,2) matrix'
wrong = np.hstack([c1, c2])
assert wrong.shape == (4,), 'hstack on 1D vectors flattens them into one long vector — that is the common trap'
print('\n✅ column_stack / stack(axis=1) give the right matrix shape; hstack flattens the result — avoid it.')


In [ ]:
def matmul(A, B):
    """Matrix × matrix — no @ allowed; call matvec on each column"""
    # ✏️ TODO: call matvec(A, col) on each column of B, and column_stack the results
    raise NotImplementedError("TODO: call matvec on each column of B, then stack into a matrix")

# Check
A = np.array([[1.,2.],[3.,4.]])
B = np.array([[5.,6.],[7.,8.]])
try:
    C = matmul(A, B)
    print('my matmul:\n', C)
    print('numpy A@B:\n', A @ B)
    assert np.allclose(C, A @ B), 'result does not match numpy'
    # Rectangular matrices: (3×2) @ (2×4) → (3×4)
    R = np.random.default_rng(0).standard_normal((3, 2))
    S = np.random.default_rng(1).standard_normal((2, 4))
    assert np.allclose(matmul(R, S), R @ S), 'matmul failed on rectangular matrices'
    print('\n✅ Passed: you built matrix multiplication yourself out of matvec (rectangular matrices included)')
except (NotImplementedError, TypeError):
    print('⬆️  Implement matmul first, then run this')


## 5. Multiplication = function composition: scale first, then rotate

The core meaning of matrix multiplication: `(A @ B) @ x` is exactly the same as `A @ (B @ x)`.
That is, applying transformation B to x and then transformation A to the result is equivalent to first **composing** B and A into one matrix `A @ B`, then applying it to x directly.

That's why matrix multiplication is **function composition**.

### Aside: think of matrices as "signal-processing boxes" — why is it called "composition"?

Imagine two machines set up as a production line: the input goes into machine B first, and what comes out goes on into machine A. `B@x` is "what signal x looks like after passing through machine B"; `A@(B@x)` is "what that intermediate result looks like after passing through machine A."

Now the question: can you weld machines B and A into one new machine that takes in x and, in a single step, produces exactly the same result as "through B, then through A"? — Yes, you can, and this "welded new machine" is precisely the matrix `A@B`. That's also why the operation "matrix multiplication" is called **function composition**: it takes the two steps "do one thing, then do another," computes them ahead of time, and packs them into a single "do it all at once" matrix.

**How does the amount of computation compare with "applying them one at a time"?** If you only need to do this for *one* vector x, "compute B@x first, then A@(...)" is usually cheaper than "compute the whole matrix A@B first, then multiply x by it" — because computing `A@B` itself takes a fair amount of work (an m×k matrix times a k×n matrix), much more expensive than doing two matvecs on one vector. But if the combination of A and B has to be applied over and over to *many* different input vectors (say, doing the "B then A" transformation on thousands of audio frames), then computing `A@B` once in advance and multiplying every vector by that single welded matrix beats multiplying twice every time — that's what "composition" means in engineering terms: pay once up front, and get efficiency back every time you reuse it.

In [ ]:
import numpy as np

# Rotation matrix R: rotate 45° counterclockwise
theta = np.pi / 4
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

# Scaling matrix S: stretch the x-axis by 2, leave y unchanged
S = np.array([[2.0, 0.0],
              [0.0, 1.0]])

x = np.array([1.0, 0.0])

# Method A: apply step by step — scale first, then rotate
step_by_step = R @ (S @ x)

# Method B: compose into one matrix, then apply
composed = (R @ S) @ x

print("scale, then rotate (step by step):", step_by_step)
print("compose, then apply (one matrix):", composed)
assert np.allclose(step_by_step, composed), "the two routes should give exactly the same result"
print("✅ (R@S)@x == R@(S@x): matrix multiplication is function composition")
print("composed matrix R@S =\n", R @ S)
print("It packs 'scale, then rotate' into a single transformation.")


### Aside: `(A@B)@x == A@(B@x)` — why does this always hold?

The code above confirmed that the two sides are equal for specific numbers, but "always holds" has to be derived directly from the definition, not taken on trust from one example. Let's follow the "row-by-row dot product" definition through:

Component i of `(A@B)@x`, by the "row-by-row dot product" definition, is the dot product of row i of `A@B` with x:

```
[(A@B)@x][i] = Σⱼ (A@B)[i,j] · x[j]
```

And `(A@B)[i,j]` is itself defined as a dot product (row i of A with column j of B): `(A@B)[i,j] = Σₖ A[i,k]·B[k,j]`. Substitute it in:

```
[(A@B)@x][i] = Σⱼ ( Σₖ A[i,k]·B[k,j] ) · x[j]
             = Σⱼ Σₖ A[i,k]·B[k,j]·x[j]
```

This is a "double sum." Swap the order of summation over j and k (group by k first, and for each k do the sum over j first):

```
             = Σₖ A[i,k] · ( Σⱼ B[k,j]·x[j] )
```

The `Σⱼ B[k,j]·x[j]` in the brackets is exactly component k of `B@x`. So:

```
[(A@B)@x][i] = Σₖ A[i,k] · (B@x)[k] = [A@(B@x)][i]
```

The two sides are equal for every component i, so the whole vectors are equal: `(A@B)@x = A@(B@x)`. This derivation uses only "matrix multiplication is defined as a sum" + "swapping the order of summation" (ordinary addition is commutative and associative), and none of the specific geometric meaning like "rotation" or "scaling" — which is why this property holds for *any* matrices, not just special cases like rotations and scalings.

## Wrap-up

You can now compute matrix multiplication yourself with `matvec(A, x)` and verify it element by element against `A @ x`. In Aurora this operation corresponds to mel filtering: an `(M, N)` matrix acting on the FFT output `(N,)` gives the mel spectrum `(M,)`. Next lesson introduces three kinds of matrices with "special powers" — orthogonal ones preserve length, symmetric ones are like mirrors, and positive-definite ones pass the energy test — the foundation for understanding eigendecomposition.

---
⬇️ **Completion check**: you've read the wrap-up; please finish the whiteboard challenge below before ticking the self-review.


## ✏️ Whiteboard challenge: matrix multiplication by hand (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: A = [[1, 2], [3, 4]], x = [5, 6]; compute A@x by hand (row-by-row dot products).

**Question 2**: A = [[2, 0], [0, 3]] (a diagonal matrix), x = [4, 5]; A@x = ?
The rule for diagonal matrices: output = diagonal entry × matching input component.

**Question 3**: Rotation matrix R = [[cos θ, -sin θ], [sin θ, cos θ]], θ = 90°,
x = [1, 0]; compute R@x by hand = ? (cos 90° = 0, sin 90° = 1)

**Question 4**: A = [[1, 0], [0, 1]] (the identity matrix), x = [7, -3]; A@x = ?
(the identity matrix is the "change nothing" transformation)

When you've finished, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1
A1 = np.array([[1.0, 2.0], [3.0, 4.0]])
x1 = np.array([5.0, 6.0])
y1_ref = np.array([17.0, 39.0])
try:
    y1 = matvec(A1, x1)
    assert np.allclose(y1, y1_ref, atol=1e-10)
    print(f"Q1 ✅  [[1,2],[3,4]] @ [5,6] = {y1}")
except (NotImplementedError, TypeError):
    print("⬜ Q1: implement matvec() first, then run the answer-check cell")

# Q2: diagonal matrix
A2 = np.array([[2.0, 0.0], [0.0, 3.0]])
x2 = np.array([4.0, 5.0])
y2_ref = np.array([8.0, 15.0])
y2 = A2 @ x2
assert np.allclose(y2, y2_ref, atol=1e-10)
print(f"Q2 ✅  diag(2,3) @ [4,5] = {y2} (a diagonal matrix scales each component)")

# Q3: 90° rotation matrix
theta = np.pi / 2
R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
x3 = np.array([1.0, 0.0])
y3 = R @ x3
y3_ref = np.array([0.0, 1.0])
assert np.allclose(y3, y3_ref, atol=1e-10)
print(f"Q3 ✅  R(90°) @ [1,0] = {np.round(y3,4)} (rotate 90° → [0,1])")

# Q4: identity matrix
I = np.eye(2)
x4 = np.array([7.0, -3.0])
y4 = I @ x4
assert np.allclose(y4, x4, atol=1e-12)
print(f"Q4 ✅  I @ [7,-3] = {y4} (the identity matrix changes nothing)")
print("\n🎉 Matrix multiplication whiteboard challenge passed! Row-by-row dot products are now second nature.")

In [ ]:
# ✏️ Self-review for this lesson
l12_review = {
    "matvec_implemented":   None,  # matvec implemented and passes its asserts? True/False
    "matmul_implemented":   None,  # matmul implemented and passes its asserts? True/False
    "transformation_intuition": None,  # understand matrix = coordinate-transformation machine? True/False
    "composition_understood": None,  # understand A@(B@x) = (A@B)@x? True/False
    "whiteboard_passed":    None,  # finished the whiteboard challenge on paper? True/False
}

unfilled = [k for k, v in l12_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l12_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L12 complete! On to L13: special matrices')

---

→ **Next lesson** · [L13 · Special matrices](L13_special_matrices.ipynb)

> Next lesson covers **special matrices**: orthogonal matrices preserve length, symmetric = mirror, and the energy test for positive-definiteness.